# **BM_25+** Retrieval

### How does __BM25+__ works?

BM25+ is a ranking algorithm used in information retrieval to score how relevant a document is to a given query. It computes a relevance score for each document by combining three main factors for each query: <br>
- __The term frequency__: a term that appears more often in a document increases the score. <br>
- __Inverse document frequency__ : a term that is rare across the collection is considered more informative and gets a higher weight. <br>
- __Document length normalization__ : the score is adjusted so that very long documents are not unfairly favored. Short documents that contain a rare term are better rewarded.

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path
import json

# Resolve project root whether launched from project root or notebooks/
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))

In [ ]:
# Import BM-25+ functions
from src.retrieval.bm25 import fit_bm25, retrieve_bm25, map_indices_to_docids

# Import data functions
from src.data.load import load_all
from src.data.preprocess import add_content_field

# Import evaluation functions
from src.evaluation.evaluate import evaluate_run, evaluate_multi_k, adapt_ground_truth

In [ ]:
# Configuration
K_VALUES = [5, 10, 20, 50]
TEXT_FIELD = "content"

RAW_DIR = Path(project_root) / "data" / "raw"
PROCESSED_DIR = Path(project_root) / "data" / "processed"

# Set to None to run on the full corpus.
MAX_DOCS = 5000

In [ ]:
# Load documents/queries with content created in notebook 01 when available.

docs_path = PROCESSED_DIR / "docs_with_content.json"
queries_path = PROCESSED_DIR / "queries_train_with_content.json"

if docs_path.exists() and queries_path.exists():
    with open(docs_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    with open(queries_path, "r", encoding="utf-8") as f:
        queries = json.load(f)
else:
    docs_raw, train_queries_raw, _, _ = load_all(RAW_DIR)
    docs, queries = add_content_field(docs_raw, train_queries_raw, clean=True)

with open(RAW_DIR / "qgts_train.json", "r", encoding="utf-8") as f:
    qgts_train = json.load(f)

gt_full = adapt_ground_truth(qgts_train)
query_ids = [str(q["id"]) for q in queries]

if MAX_DOCS is not None and len(docs) > MAX_DOCS:
    doc_by_id = {str(d["id"]): d for d in docs}
    relevant_ids = {
        doc_id
        for qid in query_ids
        for doc_id in gt_full.get(str(qid), [])
        if doc_id in doc_by_id
    }

    selected_ids = set(relevant_ids)
    if len(selected_ids) < MAX_DOCS:
        for d in docs:
            did = str(d["id"])
            if did in selected_ids:
                continue
            selected_ids.add(did)
            if len(selected_ids) >= MAX_DOCS:
                break

    docs = [d for d in docs if str(d["id"]) in selected_ids]
    gt = {
        str(qid): [doc_id for doc_id in gt_full.get(str(qid), []) if doc_id in selected_ids]
        for qid in query_ids
    }
else:
    gt = {str(qid): list(gt_full.get(str(qid), [])) for qid in query_ids}

print(f"Loaded {len(docs)} documents, {len(queries)} train queries, {len(gt)} ground-truth entries.")

Let's fit the documents using BM25+ model:

In [ ]:
start_time = time.time()
bm25plus_model = fit_bm25(docs, text_field="content", method="plus")
bm25okapi_model = fit_bm25(docs, text_field="content", method = "okapi")
fit_time = time.time() - start_time

print(f"Fit time: {fit_time:.4f} seconds")

Retrieval and Evaluation : 

We will test the retrieval for k in [5, 10, 20, 50].
By doing so, we will measure the retrieval time and also compute the metrics.

### BM25 Plus

In [ ]:
# Retrieval and evaluation for multiple k of BM25 + model
k_values = [k for k in K_VALUES if k <= len(docs)]
if not k_values:
    raise ValueError("No valid k for current number of docs.")

results_plus = []
for k in k_values:
    start_time = time.time()
    topk_indices, topk_scores = retrieve_bm25(bm25plus_model, docs, queries, k=k, text_field="content")
    retrieve_time = time.time() - start_time

    pred_docids = map_indices_to_docids(topk_indices, docs)
    eval_results = evaluate_run(pred_docids, gt, query_ids, k)

    eval_results["fit_time_s"] = fit_time
    eval_results["retrieve_time_s"] = retrieve_time
    eval_results["avg_retrieve_time_ms"] = (retrieve_time / len(queries)) * 1000
    results_plus.append(eval_results)

    print(f"--- Results for k={k} ---")
    print(f"Retrieve time: {retrieve_time:.4f} s (avg {eval_results['avg_retrieve_time_ms']:.2f} ms/query)")
    print(f"Top-k indices shape: {topk_indices.shape}")
    print(f"Top-k scores shape: {topk_scores.shape}")
    print("Evaluate run:", eval_results)

### BM25 Okapi

In [ ]:
# Retrieval and evaluation for multiple k of BM25 Okapi model
k_values = [k for k in K_VALUES if k <= len(docs)]
if not k_values:
    raise ValueError("No valid k for current number of docs.")

results_okapi = []
for k in k_values:
    start_time = time.time()
    topk_indices, topk_scores = retrieve_bm25(bm25okapi_model, docs, queries, k=k, text_field="content")
    retrieve_time = time.time() - start_time

    pred_docids = map_indices_to_docids(topk_indices, docs)
    eval_results = evaluate_run(pred_docids, gt, query_ids, k)

    eval_results["fit_time_s"] = fit_time
    eval_results["retrieve_time_s"] = retrieve_time
    eval_results["avg_retrieve_time_ms"] = (retrieve_time / len(queries)) * 1000
    results_okapi.append(eval_results)

    print(f"--- Results for k={k} ---")
    print(f"Retrieve time: {retrieve_time:.4f} s (avg {eval_results['avg_retrieve_time_ms']:.2f} ms/query)")
    print(f"Top-k indices shape: {topk_indices.shape}")
    print(f"Top-k scores shape: {topk_scores.shape}")
    print("Evaluate run:", eval_results)

In [ ]:
print("Results of BM25 Plus model")
results_plus_df = pd.DataFrame(results_plus).sort_values("k").reset_index(drop=True)
display(results_plus_df)

print("Results of BM25 Okapi model")
results_okapi_df = pd.DataFrame(results_okapi).sort_values("k").reset_index(drop=True)
display(results_okapi_df)



Summary table:

In [ ]:
# Coherent multi-k evaluation from one retrieval run at max(k)
k_max = max(k_values)
topk_indices_max, topk_scores_max = retrieve_bm25(bm25plus_model, docs, queries, k=k_max, text_field="content")
pred_docids_max = map_indices_to_docids(topk_indices_max, docs)

multi_results = evaluate_multi_k(pred_docids_max, gt, query_ids, k_values)
print("\nMulti-k Results of Plus Model:")
display(multi_results)

In [ ]:
# Coherent multi-k evaluation from one retrieval run at max(k)
k_max = max(k_values)
topk_indices_max, topk_scores_max = retrieve_bm25(bm25okapi_model, docs, queries, k=k_max, text_field="content")
pred_docids_max = map_indices_to_docids(topk_indices_max, docs)

multi_results = evaluate_multi_k(pred_docids_max, gt, query_ids, k_values)
print("\nMulti-k Results of Okapi Model:")
display(multi_results)

### Comparison with TF-IDF model

In [ ]:
# Read TF-IDF results
tfidf_results_df = pd.read_csv(project_root / "outputs" / "runs" / "tfidf_results.csv")
tfidf_results_df = tfidf_results_df.sort_values("k").reset_index(drop=True)
print("TF-IDF model results")
display(tfidf_results_df)

# Prepare BM25 Plus and Okapi results
results_plus_df["model"] = "BM25 Plus"
results_okapi_df["model"] = "BM25 Okapi"

# Combine all three models into a tidy table
models_dfs = [tfidf_results_df.copy() for _ in range(2)]
models_dfs[0]["model"] = "TF-IDF"
models_dfs[1] = results_plus_df.copy()
models_dfs[1]["model"] = "BM25 Plus"
models_dfs.append(results_okapi_df.copy())
comparative_long_df = pd.concat(models_dfs, ignore_index=True)

# Select main columns for display
columns = ["model", "k", "precision@k", "recall@k", "mrr@k", "fit_time_s", "retrieve_time_s", "avg_retrieve_time_ms"]
comparative_long_df = comparative_long_df[columns]

# Display clear comparative table without index
print("Comparative table of the three models (TF-IDF, BM25 Plus, BM25 Okapi):")
display(comparative_long_df.sort_values(["k", "model"]).reset_index(drop=True))


**Latency and Quality Trade-off**

- **BM25 vs TF-IDF Latency:**
  - BM25 (both Plus and Okapi) is generally slower than TF-IDF due to its more complex scoring mechanism, but the difference is often acceptable for offline or batch retrieval tasks.
  - TF-IDF is faster and can be preferable when very low latency is required.

- **Precision and Recall Evolution with k:**
  - As `k` increases, recall improves for all models, since more relevant documents are likely to be retrieved.
  - Precision tends to decrease with higher `k`, as more non-relevant documents are included in the top-k results.
  - MRR (Mean Reciprocal Rank) usually decreases slightly as `k` increases, since the first relevant document may appear later in the ranking.

- **Trade-off:**
  - BM25 often provides better retrieval quality (higher precision/recall) than TF-IDF, especially for smaller `k`.
  - The choice between BM25 and TF-IDF depends on the application: if quality is more important than speed, BM25 is preferable; if speed is critical and quality is less important, TF-IDF may suffice.